In [98]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import seaborn as sns
import xarray
import torch
import numpy as np

import jax_cfd.base as cfd
# import jax_cfd.base.forcings as forcings
import jax_cfd.base.grids as grids
import jax_cfd.spectral as spectral

# from jax_cfd.base import boundaries
# from jax_cfd.base import forcings
# from jax_cfd.base import grids
from jax_cfd.spectral import forcings as spectral_forcings
from jax_cfd.spectral import time_stepping
from jax_cfd.spectral import types
from jax_cfd.spectral import utils as spectral_utils

from jax.experimental.host_callback import call

from functools import partial

from typing import Callable, Optional, Tuple
Array = grids.Array
GridArrayVector = grids.GridArrayVector
GridVariableVector = grids.GridVariableVector
ForcingFn = Callable[[GridVariableVector], GridArrayVector]

import dataclasses

import os

from jax import config
# config.update("jax_enable_x64", True)


In [99]:
class NavierStokes2D:
    """Breaks the Navier-Stokes equation into implicit and explicit parts.

    Implicit parts are the linear terms and explicit parts are the non-linear
    terms.

    Attributes:
    viscosity: strength of the diffusion term
    grid: underlying grid of the process
    smooth: smooth the advection term using the 2/3-rule.
    forcing_fn: forcing function, if None then no forcing is used.
    drag: strength of the drag. Set to zero for no drag.
    """

    def __init__(self,viscosity: float,
                 grid: grids.Grid,
                 drag: float = 0.,
                 dt: float = 1e-3,
                 noise_scale: float = 1e-2,
                 smooth: bool = True,):
        
        self.viscosity = viscosity
        self.grid = grid
        self.drag = drag
        self.dt = dt
        self.noise_scale = noise_scale
        self.smooth = smooth
        
        self.kx, self.ky = self.grid.rfft_mesh()
        self.laplace = (jnp.pi * 2j)**2 * (self.kx**2 + self.ky**2)
        self.filter_ = spectral_utils.brick_wall_filter_2d(self.grid)
        self.linear_term = self.viscosity * self.laplace - self.drag
        self.offsets = (0,0)
        self.x = self.grid.mesh(self.offsets)[0]
        self.y = self.grid.mesh(self.offsets)[1]

    # setup the forcing function with the caller-specified grid.
    
    @partial(jax.jit, static_argnums=0)
    def explicit_terms(self, vorticity_hat, key):
        velocity_solve = spectral_utils.vorticity_to_velocity(self.grid)
        vxhat, vyhat = velocity_solve(vorticity_hat)
        vx, vy = jnp.fft.irfftn(vxhat), jnp.fft.irfftn(vyhat)

        grad_x_hat = 2j * jnp.pi * self.kx * vorticity_hat
        grad_y_hat = 2j * jnp.pi * self.ky * vorticity_hat
        grad_x, grad_y = jnp.fft.irfftn(grad_x_hat), jnp.fft.irfftn(grad_y_hat)

        advection = -(grad_x * vx + grad_y * vy)
        advection_hat = jnp.fft.rfftn(advection)
        
        randn_noise = self.noise_scale*jax.random.normal(key = key,shape = (4,))/jnp.sqrt(self.dt)
        
        vorticity_noise = randn_noise[0]*jnp.sin(6*self.x) + randn_noise[1]*jnp.cos(7*self.x) \
                          + randn_noise[2]*jnp.sin(5*(self.x+self.y)) + randn_noise[3]*jnp.cos(8*(self.x+self.y))
        
        vorticity_noise_hat = jnp.fft.rfft2(vorticity_noise)

        if self.smooth is not None:
            advection_hat *= self.filter_

        terms = advection_hat + vorticity_noise_hat

        return terms
    
    @partial(jax.jit, static_argnums=0)
    def implicit_terms(self, vorticity_hat):
        return self.linear_term * vorticity_hat
    
    @partial(jax.jit, static_argnums=0)
    def implicit_solve(self, vorticity_hat, time_step):
        return 1 / (1 - time_step * self.linear_term) * vorticity_hat
    
    @partial(jax.jit, static_argnums=0)
    def euler_step(self,vorticity_hat,key):
        
        g = vorticity_hat + self.dt*self.explicit_terms(vorticity_hat,key)
        u1 = self.implicit_solve(g, self.dt)
        
        return u1
        

In [102]:
%%time 

viscosity = 1e-3
rand_seed = 10
dt = 1e-4
key = jax.random.PRNGKey(rand_seed)
print(key)
grid = grids.Grid((128,128), domain=((0, 2 * jnp.pi), (0, 2 * jnp.pi)))
smooth = True
equation = NavierStokes2D(viscosity,grid,drag=-0.1,dt = dt,smooth=smooth)

v0 = cfd.initial_conditions.filtered_velocity_field(jax.random.PRNGKey(1), grid, 7, 2)
vorticity0 = cfd.finite_differences.curl_2d(v0).data
vorticity_hat0 = jnp.fft.rfftn(vorticity0)

for i in range(1000):
    key,_ = jax.random.split(key)
#     print(key)
    vorticity_hat1 = equation.euler_step(vorticity_hat0,key)
    vorticity_hat0 = vorticity_hat1
    
vorticity1 = jnp.fft.irfft2(vorticity_hat0)

[ 0 10]
CPU times: user 830 ms, sys: 115 ms, total: 945 ms
Wall time: 636 ms


# Forced Turbulence

Kolmogorov forcing with some randomness.

In [ ]:
%%time 

def kolmogorov_data_generation(
    grid_size: int = 128,
    noise_scale: float = 2e-1,
    num_trajectories: int = 10,
    viscosity: float = 1e-3,
    max_velocity: float = 7.0,
    max_courant_number: float = 0.5,
    final_time: float = 10,
    time_snapshots: int = 1,
) -> Tuple[torch.tensor,torch.tensor,torch.tensor,torch.tensor]:


    grid = grids.Grid((grid_size, grid_size), domain=((0, 2 * jnp.pi), (0, 2 * jnp.pi)))
    
    dt = cfd.equations.stable_time_step(max_velocity, max_courant_number, viscosity, grid)
    
    dt = 1e-4

    inner_steps = int((final_time // dt) // time_snapshots)
    
    smooth = True # use anti-aliasing 

    data_all = jnp.zeros((num_trajectories,time_snapshots,grid_size,grid_size))

    v0 = cfd.initial_conditions.filtered_velocity_field(jax.random.PRNGKey(0), grid, max_velocity, 2)
    vorticity0 = cfd.finite_differences.curl_2d(v0).data
    
    vorticity0 = data_all_jax[18,0,...]

    vorticity_hat0 = jnp.fft.rfftn(vorticity0)
    
    equation = NavierStokes2D(viscosity,grid,drag=-0.1,  dt = dt, noise_scale = noise_scale, smooth=smooth)
    
    for i in range(num_trajectories):
        
        key = jax.random.PRNGKey(i)
        vorticity_hat_curr = vorticity_hat0
        
        for j in range(time_snapshots):
            
            for steps in range(inner_steps):
                
                key,_ = jax.random.split(key)

                vorticity_hat_next = equation.euler_step(vorticity_hat_curr,key)
                vorticity_hat_curr = vorticity_hat_next

            trajectory_real = jnp.fft.irfft2(vorticity_hat_curr).squeeze()

        data_all = data_all.at[i,j,...].set(trajectory_real)

    data_jax = data_all
    data_torch = torch.from_numpy(np.asarray(data_all).copy())
    
#     time_jax = dt * jnp.arange(time_snapshots) * inner_steps
#     time_torch = torch.from_numpy(np.asarray(time_jax).copy())
    
    return data_jax, data_torch

data_jax, data_torch = kolmogorov_data_generation(grid_size = 128)

data = data_torch


CPU times: user 5min 12s, sys: 1min 40s, total: 6min 52s
Wall time: 4min 26s
